In [2]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import torch
from datasets import Dataset
import pandas as pd
from torch.utils.data import DataLoader
from evaluate import load
from tqdm import tqdm
import os
import argparse
import yaml

# bertscore = load("bertscore")
# model_id = "meta-llama/Llama-3.2-1B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token_id = tokenizer.eos_token_id
# model = AutoModelForCausalLM.from_pretrained("/home/station_06/DATA01/sentimatic-outputs/llama3.2-inst-negative-re/checkpoint-1425")
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

# pipe = pipeline(
#     "text-generation",
#     model="/home/station_06/DATA01/sentimatic-outputs/llama3.2-inst-negative-re/checkpoint-760",
#     device_map="auto",
# )
# messages = [
#     {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
#     {"role": "user", "content": "Who are you?"},
# ]
# outputs = pipe(
#     messages,
#     max_new_tokens=64,
# )
# print(outputs[0]["generated_text"][-1])


In [34]:
# pip install -q transformers
from transformers import pipeline

checkpoint = "MBZUAI/LaMini-Flan-T5-783M"

model = pipeline('text2text-generation', model = checkpoint)

input_prompt = "'Generate an agent's response to the following customer message: \n Customer: Help me"
generated_text = model(input_prompt, max_length=512, do_sample=True)[0]['generated_text']

print("Response", generated_text)


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Response Sure, could you please provide me with details about what is needed?


In [2]:
test_file = os.path.join('/home/station_06/Sentimatic/dataset/Tweetsumm_sentiment_analysis/test/negative_test.csv')
test_df = pd.read_csv(test_file)[['customer_01', 'agent']]

In [3]:
# responses = []
# for idx, customer in tqdm(enumerate(test_df["customer_01"])):
#     messages = [
#         {"role": "system", "content": "You are a customer service chatbot. Generate a agent's response to the following customer message."},
#         {"role": "user", "content": customer},
#     ]
#     outputs = pipe(
#         messages,
#         max_new_tokens=256,
#         min_length=10,
        
#     )
#     responses.append(outputs[0]["generated_text"][-1]['content'])
#     print(f"Customer: {customer}")
#     print(f"Agent: {responses[-1]}")
#     print("="*50)

# test_df['llama_res'] = responses
# test_df.to_csv('llama_res.csv', index=False)

In [4]:
def clean_output(output):
    return tokenizer.decode(output[0], skip_special_tokens=True).split("assistant\n\n")[-1].split("https:")[0]

def compute_bertscore(predictions, references, lang="en"):
    results = bertscore.compute(predictions=predictions, references=references, lang=lang)
    return results["f1"]

In [ ]:
responses = []
num_return_sequences = 3

beam_outputs = []
top_k_outputs = []
top_p_outputs = []

for idx, customer in tqdm(enumerate(test_df["customer_01"])):
    messages = [
        {"role": "system", "content": "You are a customer service chatbot. Generate a agent's response to the following customer message."},
        {"role": "user", "content": customer},
    ]
    input_prompt = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    
    padded_inputs = torch.nn.utils.rnn.pad_sequence(input_prompt, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = (padded_inputs != tokenizer.pad_token_id).long()
    
    model_inputs = {
        "input_ids": padded_inputs.to(device),
        "attention_mask": attention_mask.to(device),
    }
    
    for i in range(3):
        with torch.no_grad():
            if i == 0:
                output0 = model.generate(model_inputs["input_ids"], attention_mask=model_inputs["attention_mask"], num_beams=num_return_sequences, num_return_sequences=num_return_sequences, early_stopping=True,
                                        max_length=200, temperature=0.9)
            if i == 1:
                output1 = model.generate(model_inputs["input_ids"], attention_mask=model_inputs["attention_mask"], top_k=50, do_sample=True, num_return_sequences=num_return_sequences, early_stopping=True,
                                        max_length=200, temperature=0.9)
            elif i == 2:
                output2 = model.generate(model_inputs["input_ids"], attention_mask=model_inputs["attention_mask"], top_p=0.9, do_sample=True, num_return_sequences=num_return_sequences, early_stopping=True,
                                        max_length=200, temperature=0.9)
    
    beam_outputs.append(clean_output(output0))
    top_k_outputs.append(clean_output(output1))
    top_p_outputs.append(clean_output(output2))
    
    print("="*50)
    print(f"Customer: {customer}")
    print(f"Agent (beam search): {beam_outputs[-1]}")
    print(f"Agent (top-k sampling): {top_k_outputs[-1]}")
    print(f"Agent (top-p sampling): {top_p_outputs[-1]}")
    print("="*50)
    

beam_score = compute_bertscore(beam_outputs, test_df["agent"].tolist())
top_k_score = compute_bertscore(top_k_outputs, test_df["agent"].tolist())
top_p_score = compute_bertscore(top_p_outputs, test_df["agent"].tolist())

test_df['beam_res'] = beam_outputs
test_df['top_k_res'] = top_k_outputs
test_df['top_p_res'] = top_p_outputs

test_df['beam_score'] = beam_score
test_df['top_k_score'] = top_k_score
test_df['top_p_score'] = top_p_score

test_df.to_csv('llama_res.csv', index=False)